In [1]:
# Caregando as bibs

import pandas as pd
import re # expressões regulates (para limpar texto)
from pathlib import Path

CAMINHO = R"C:\turma-visualizacao-de-dados\alunos\sergio_leite\Mini_projeto_Avaliativo_01\base_varejo.csv"

# pd.read_csv vai ler e add os dados de leitura a variavel df que será usada para acessar o arquivo e gerar dados

df = pd.read_csv(
    CAMINHO,
    sep=';', # padrão de ; para separar colunas e não virgula
    encoding='cp1252', # encoding do windows para leitura de acentos
    decimal=',' # add padrão de virgula para separar decimais
)


In [3]:
print(df.head()) # verificando as primeiras linhas
print(df.tail()) # verificando as últimas linhas
print()
print(f'O DF possui {df.shape[0]} Linhas.') # Verificando a quantidade de linhas
print(f'O DF possui {df.shape[1]} Colunas') # verificando a quantidade de colunas
print()
df.info()


         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT  \
0  01/02/2019   1000    534         M      4       1      C     67    BEBIDAS   
1  01/02/2019   1000    534         M      4       1      C     70    BEBIDAS   
2  01/02/2019   1000    534         M      4       1      C    178    HIGIENE   
3  01/02/2019   1000    534         M      4       1      C      4  ALIMENTOS   
4  01/02/2019   1000    534         M      4       1      C    175    LIMPEZA   

                PR_NOME  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  
0  REFRIGERANTE GUARANA          NaN          NaN          NaN          NaN  
1   REFRIGERANTE OUTROS          NaN          NaN          NaN          NaN  
2       LENCO UMEDECIDO          NaN          NaN          NaN          NaN  
3               ABACAXI          NaN          NaN          NaN          NaN  
4     LIMPADOR MULTIUSO          NaN          NaN          NaN          NaN  
              DATA   CO_ID  CL_ID CL_GENERO  

In [4]:
print('Valores nulos por coluna')
# df.isnull cria uma tabela true / false onde os true são os nulos
print(df.isnull().sum()) # calculando a soma de valores nulos por coluna
print()
print('Quantidade de linhas clonadas identicas')
print(df.duplicated().sum()) # vai encontrar linhas clonadas que sejam identicas
print()
print('Verificando o formato da coluna DATA')
print(df.DATA.head(5)) # reportando a coluna DATA em formato invalido. Está em formato objeto


Valores nulos por coluna
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64

Quantidade de linhas clonadas identicas
96553

Verificando o formato da coluna DATA
0    01/02/2019
1    01/02/2019
2    01/02/2019
3    01/02/2019
4    01/02/2019
Name: DATA, dtype: object


In [5]:
# importação da biblioteca
from datetime import datetime, timedelta

# Convertendo toda a coluna data para DATETIME
df['DATA'] = pd.to_datetime(df['DATA'], format= '%d/%m/%Y') # O format= vai definir o formato que queremos as datas
print('O novo formato da coluna DATA:')
print(df['DATA'].dtype)
print('Seguem as primeiras linhas da coluna DATA:')
print(df.DATA.head())

# Convertendo a coluna DATA de TEXTO para DATA - strptime
# o primeiro valor data será o exemplo
# data_txt = '01/02/2019' #criamos a var para receber o datetime com o modelo do texto e o formato da data
# data_obj = datetime.strptime(data_txt, '%d/%m/%Y') # %d/%m/%Y
# print(f'Data em String (texto) original: {data_txt}')
# print(f'Objeto datetime: {data_obj}')
# print(f'O tipo da data é: {type(data_obj)}')
# Ia usar essa solução, mas entendi que ela precisa ser usada linha por linha e vai deixar o código lento.



O novo formato da coluna DATA:
datetime64[ns]
Seguem as primeiras linhas da coluna DATA:
0   2019-02-01
1   2019-02-01
2   2019-02-01
3   2019-02-01
4   2019-02-01
Name: DATA, dtype: datetime64[ns]


In [7]:
# Criando uma cópia do df para fazer as modificações sem mexer no original
df_limpo = df.copy()

# removendo as duplicadas com o drop.duplicates()
df_limpo = df_limpo.drop_duplicates() 

# Testando o df original e o df limpo para saber quantas linhas foram retiradas
print(df.shape[0])
print(df_limpo.shape[0])
print(f'Foram retiradas {df.shape[0] - df_limpo.shape[0]} linhas duplicadas do DF.')

830000
733447
Foram retiradas 96553 linhas duplicadas do DF.


In [ ]:
df_limpo['MES'] = df_limpo['DATA'].dt.month
df_limpo['ANO'] = df_limpo['DATA'].dt.year

# print(df_limpo['MES'])
# print(df_limpo['ANO'])

vendas_por_mes = df_limpo.groupby('MES').size() # conta o numero de linhas em cada mês
print('Quantidade de vendas por mês:')
print(vendas_por_mes)

Quantidade de vendas por mês:
MES
1     74121
2     67130
3     56929
4     59220
5     72799
6     59848
7     60936
8     58022
9     60687
10    65350
11    36075
12    62330
dtype: int64


In [14]:
# A Escolha por imputar nulos por considerar que os dados de outras colunas podem ser úteis.
# df.fillna() # preencher/imputar a mensagem desejada onde ele encontrar valores nulos NaN.
# atribuimos a coluna PR_CAT acessada para gravar a informação no DF_LIMPO.

df_limpo['PR_CAT'] = df_limpo['PR_CAT'].fillna('Sem categoria') # vai prerncher a mensagem Sem categoria onde for NaN
print(f'A quantidade de campos nulos na coluna PR-CAT foi: {df_limpo['PR_CAT'].isnull().sum()}')

A quantidade de campos nulos na coluna PR-CAT foi: 0


In [19]:
# Tratamento de Nulos e Condicionais (N3)
# usando if/elif como pede o critério.

for coluna in df_limpo.columns:
    if df_limpo[coluna].isnull().sum() == 0:
        continue
    elif df_limpo[coluna].dtype == 'object':
        # colunas de texto: preenche com um rótulo padrão
        df_limpo[coluna] = df_limpo[coluna].fillna('Não informado')
    elif df_limpo[coluna].dtype in ['int64', 'float64']:
        # colunas numéricas: preenche com a mediana, mais robusta a outliers que a média
        df_limpo[coluna] = df_limpo[coluna].fillna(df_limpo[coluna].median())

print('Valores nulos após o tratamento condicional:')
print(df_limpo.isnull().sum())

C:\Users\android\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\android\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\android\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\android\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: Runti

Valores nulos após o tratamento condicional:
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    733447
Unnamed: 11    733447
Unnamed: 12    733447
Unnamed: 13    733447
MES                 0
ANO                 0
dtype: int64


In [ ]:
# Estatísticas em uma coluna numérica
# .mean() - média
# .min() - menor valor
# .max() - maior valor
# .median() - mediana - o valor do meio quando ordenado

print(f'A média de filhos dos clientes (Coluna CL_FHL) é: {df_limpo['CL_FHL'].mean().round(2)}')
print()
print(f'A mediana (valor do meio, quando ordenado) da coluna filhos dos clientes (Coluna CL_FHL) é: {df_limpo['CL_FHL'].median().round(2)}')
print()

# O describe() é muito usado para ver a estatistica descritiva das colunas numéricas 
# ajuda a confirmar que dados estão no formato esperado, sem valores absurdos e sem erros de leituras ou outros erros.
# count - quantos valores não nulos existem
# mean - média
# std - Desvio padrão - o quanto os valores variam em torno da média
# min - Menor valor
# 25% - 25% dos registros estão abaixo desse valor
# 50% - mediana. 50% dos registros abaixo de 50%
# 75% - 75% dos registros estão abaixo desse valor
# max - Maior valor
# mode - (o valor que mais se repete) na coluna

print(f'O Desvio padrão da coluna CL_FHL é: {df_limpo['CL_FHL'].std().round(2)}')
print()
print(f'O Valor Mínimo da coluna CL_FHL é: {df_limpo['CL_FHL'].min().round(2)}')
print()
print(f'O Valor Máximo da coluna CL_FHL é: {df_limpo['CL_FHL'].max().round(2)}')
print()
print(f'A Quantidade de valores NÃO NULOS (registros Válidos) da coluna CL_FHL é: {df_limpo['CL_FHL'].count().round(2)}')
print()
print(f'O número de filhos (coluna CL_FHL) mais comum entre os clientes é de: {df_limpo['CL_FHL'].mode()[0]}') # colocar o [0] para imprimir apenas o núumero da primeira posição (indice [0]). pois sem isso, o pandar imprime outros números que estiverem empatados na quantidade de filhos

A média de filhos dos clientes (Coluna CL_FHL) é: 1.15

A mediana (valor do meio, quando ordenado) da coluna filhos dos clientes (Coluna CL_FHL) é: 0.0

O Desvio padrão da coluna CL_FHL é: 1.42

O Valor Mínimo da coluna CL_FHL é: 0

O Valor Máximo da coluna CL_FHL é: 4

A Quantidade de valores NÃO NULOS (registros Válidos) da coluna CL_FHL é: 733447

O número de filhos (coluna CL_FHL) mais comum entre os clientes é de: 0



In [ ]:
# .groupby('coluna') serve para agrupar/juntar a tabela com base em categorias
# (ex: junta todas as linhas que têm a mesma categoria de produto ou mesma data)
# size() faz a contagem de linhas em cada uma das pilhas que o groupby faz. Ex.: mulheres que compraram refri ele vai contar e passar o numero
# o sort.values() vai organizar o código em ordem
# ascending=False - ascendente falso, para quando nao queremos ascendente. No nosso caso vai aparecer decrescente

analise_vendas_genero_categoria = df_limpo.groupby(['CL_GENERO', 'PR_CAT']).size().sort_values(ascending=False)
print(analise_vendas_genero_categoria)

analise_vendas_genero_categoria = df_limpo.groupby(['CL_GENERO', 'PR_CAT']).size().sort_values(ascending=False)
print(analise_vendas_genero_categoria)




CL_GENERO  PR_CAT    
F          ALIMENTOS     200274
M          ALIMENTOS     183923
F          HIGIENE        71721
           LIMPEZA        67328
M          HIGIENE        65981
           LIMPEZA        61304
F          BEBIDAS        19764
M          BEBIDAS        18500
F          PET            14809
M          PET            13744
F          ACESSORIOS      6839
M          ACESSORIOS      6032
F          #N/D            1692
M          #N/D            1536
dtype: int64


In [20]:
# Manipulação de Arquivos CSV (N3)
# Exportando a base já limpa (duplicatas removidas, nulos tratados, colunas MES/ANO adicionadas)

df_limpo.to_csv('base_varejo_limpa.csv', index=False, sep=';', encoding='cp1252')
print('Arquivo base_varejo_limpa.csv exportado com sucesso.')

Arquivo base_varejo_limpa.csv exportado com sucesso.


Principais insights obtidos:

- A maioria dos compradores não tem filhos
- A Categoria Alimentos é o item muito mais consumido em relação a diferença dos outros
- Mulheres sempre compram mais que homens, em todas as categorias as mulheres compraram mais que os homens
- Em todas as categorias, homens e mulheres compram proporcionalmente na mesma ordem de prioridades
